# Sonic Drive-In Menu Ingestion to Azure AI Search

## 1: Notebook Introduction
This notebook demonstrates how to:
1. Configure Azure OpenAI and Azure AI Search services for the Sonic Drive-In menu.
2. Prepare the Sonic menu JSON data for ingestion into Azure AI Search.
3. Upload the prepared data to Azure AI Search for hybrid semantic search capabilities.


## 2: Install Required Packages

### Description
This cell installs all the necessary packages required for the notebook.
It ensures that all dependencies are met before proceeding with the rest of the notebook.


In [ ]:
%pip install azure-core azure-search-documents python-dotenv langchain-openai langchain-community openai==1.54.3 "httpx>=0.23.0,<0.28.0" pydantic tenacity

## 3: Imports and Environment Setup

### Description
This cell imports necessary libraries and loads environment variables using `dotenv`.
Ensure your `.env` file is properly set up with the required Azure API keys and endpoints.


In [ ]:
# Import required libraries
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch
)
from dotenv import load_dotenv
from openai import AzureOpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

import json
import os
import re

# Load environment variables from the repo backend .env (fallback to current dir)
env_path = os.path.join('..', 'app', 'backend', '.env')
load_dotenv(dotenv_path=env_path)
load_dotenv()  # allows overriding with local .env if present
print(f'Loaded env file at {env_path}: {os.path.exists(env_path)}')


## 4: Azure OpenAI and Azure AI Search Configuration

### Description
This cell sets up the Azure OpenAI and AI Search configurations.
Ensure that the endpoints, API keys, and deployment names in the `.env` file match your Azure resource setup.

**Note:** This notebook uses the `sonic-menu-items` index name, distinct from other indexes in the project.


In [ ]:
# Quick credential sanity check (prints masked values)
def _mask(val: str):
    if not val:
        return '<missing>'
    return f'len={len(val)}, suffix={val[-4:]}'

vars_to_check = {
    'AZURE_OPENAI_EASTUS_ENDPOINT': os.getenv('AZURE_OPENAI_EASTUS_ENDPOINT'),
    'AZURE_OPENAI_EASTUS_API_KEY': os.getenv('AZURE_OPENAI_EASTUS_API_KEY') or os.getenv('AZURE_OPENAI_API_KEY'),
    'AZURE_OPENAI_API_VERSION': os.getenv('AZURE_OPENAI_API_VERSION'),
    'AZURE_OPENAI_EMBEDDING_DEPLOYMENT': os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT'),
    'AZURE_SEARCH_ENDPOINT': os.getenv('AZURE_SEARCH_ENDPOINT'),
    'AZURE_SEARCH_API_KEY': os.getenv('AZURE_SEARCH_API_KEY'),
}

# Normalize to expected variable if only generic name is present
if not os.getenv('AZURE_OPENAI_EASTUS_API_KEY') and os.getenv('AZURE_OPENAI_API_KEY'):
    os.environ['AZURE_OPENAI_EASTUS_API_KEY'] = os.getenv('AZURE_OPENAI_API_KEY')

for name, value in vars_to_check.items():
    print(f'{name}: {_mask(value)}')

missing = [k for k, v in vars_to_check.items() if not v]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")
else:
    print('All required environment variables are present.')


In [ ]:
# Azure OpenAI setup
aoai_eastus_endpoint = os.getenv('AZURE_OPENAI_EASTUS_ENDPOINT')
aoai_eastus_api_key = os.getenv('AZURE_OPENAI_EASTUS_API_KEY')
aoai_openai_api_version = os.getenv('AZURE_OPENAI_API_VERSION')
aoai_embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')

# Initialize the Azure OpenAI client
aoai_client = AzureOpenAI(
    azure_endpoint=aoai_eastus_endpoint,
    api_version=aoai_openai_api_version,
    api_key=aoai_eastus_api_key,
)

# Sonic Drive-In search index name
index_name = 'sonic-menu-items'

# Azure AI Search credentials
search_service_endpoint = os.getenv('AZURE_SEARCH_ENDPOINT')
search_api_key = os.getenv('AZURE_SEARCH_API_KEY')
search_client = SearchClient(endpoint=search_service_endpoint, index_name=index_name, credential=AzureKeyCredential(search_api_key))
search_index_client = SearchIndexClient(endpoint=search_service_endpoint, credential=AzureKeyCredential(search_api_key))

print(f'Azure OpenAI and Azure Search clients initialized successfully.')
print(f'Target index: {index_name}')


## 5: Define and Create/Update Index Schema with Semantic Configuration

### Description
This cell defines the schema for the Sonic Drive-In Azure AI Search index, including fields for semantic search and vector search capabilities. It deletes any existing index with the same name and creates a fresh one.


### Note on enabling semantic ranker

Semantic search is a service-level capability. If you see errors like "semantic search is not enabled," ensure the service has semantic ranker turned on (Standard tier or higher). You can enable it via CLI if the portal UI does not surface a toggle:

`az search service update --name <service> --resource-group <rg> --semantic-search standard`

Once enabled, the existing index schema here (with `semantic_search` and `menuSemanticConfig`) works without further changes.


In [ ]:
# Define and Create/Update Index Schema with Semantic Configuration
if not index_name:
    raise RuntimeError('index_name is not set.')

index_schema = SearchIndex(
    name=index_name,
    fields=[
        SimpleField(name='id', type=SearchFieldDataType.String, key=True, sortable=True, filterable=True),
        SearchField(name='category', type=SearchFieldDataType.String, sortable=True, filterable=True, facetable=True),
        SearchField(name='name', type=SearchFieldDataType.String, sortable=True, filterable=True, facetable=True),
        SearchField(name='description', type=SearchFieldDataType.String),
        SearchField(name='longDescription', type=SearchFieldDataType.String),
        SearchField(name='origin', type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchField(name='caffeineContent', type=SearchFieldDataType.String, filterable=True),
        SearchField(name='brewingMethod', type=SearchFieldDataType.String, filterable=True),
        SearchField(name='popularity', type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchField(name='sizes', type=SearchFieldDataType.String, filterable=False, facetable=False),
        SearchField(
            name='embedding',
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=3072,
            vector_search_profile_name='menuHnswProfile'
        )
    ],
    vector_search=VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name='menuHnsw',
                kind='hnsw',
                parameters={'m': 10, 'efConstruction': 200}
            )
        ],
        profiles=[
            VectorSearchProfile(
                name='menuHnswProfile',
                algorithm_configuration_name='menuHnsw',
                vectorizer_name='menuVectorizer'
            )
        ],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name='menuVectorizer',
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=aoai_eastus_endpoint,
                    deployment_name=aoai_embedding_deployment,
                    model_name=aoai_embedding_deployment,
                    api_key=aoai_eastus_api_key
                )
            )
        ]
    ),
    semantic_search=SemanticSearch(
        configurations=[
            SemanticConfiguration(
                name='menuSemanticConfig',
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name='name'),
                    content_fields=[
                        SemanticField(field_name='description'),
                        SemanticField(field_name='longDescription'),
                        SemanticField(field_name='category')
                    ]
                )
            )
        ]
    ),
)

# Delete the existing index if it exists
try:
    search_index_client.delete_index(index_name)
    print(f'Deleted existing index: {index_name}')
except Exception as e:
    print(f'Index {index_name} does not exist or could not be deleted: {e}')

# Create the index schema
try:
    search_index_client.create_or_update_index(index=index_schema)
    print(f'Created index: {index_name}')
except Exception as e:
    raise RuntimeError(f'Failed to create/update index {index_name}: {e}')

# Verify index exists
existing = [i.name for i in search_index_client.list_indexes()]
print(f'Indexes present: {existing}')
if index_name not in existing:
    raise RuntimeError(f'Index {index_name} was not found after creation.')


## 6: Load and Process Sonic Drive-In Menu Data

### Description
This cell reads the `sonicmenuitems.json` file containing the Sonic Drive-In menu, processes the data to ensure all fields are populated, and prints the structured JSON data.


In [ ]:
# Define candidate paths to the Sonic menu data file
menu_items_candidates = [
    os.path.join('..', 'app', 'frontend', 'src', 'data', 'sonicmenuitems.json'),
]

menu_items_path = None
for candidate in menu_items_candidates:
    if os.path.exists(candidate):
        menu_items_path = candidate
        break

if not menu_items_path:
    raise FileNotFoundError(f'Could not find sonicmenuitems.json. Tried: {menu_items_candidates}')

print(f'Using sonicmenuitems.json at: {menu_items_path}')

# Read the JSON file
with open(menu_items_path, 'r') as file:
    menu_items = json.load(file)

# Build JSON with all fields, put empty string if not applicable
structured_menu_items = {'menuItems': []}

for category in menu_items['menuItems']:
    category_dict = {'category': category['category'], 'items': []}
    for item in category['items']:
        item_dict = {
            'name': item['name'],
            'description': item.get('description', ''),
            'longDescription': item.get('longDescription', ''),
            'origin': item.get('origin', ''),
            'caffeineContent': item.get('caffeineContent', ''),
            'brewingMethod': item.get('brewingMethod', ''),
            'popularity': item.get('popularity', ''),
            'sizes': [{'size': size['size'], 'price': size['price']} for size in item.get('sizes', [])]
        }
        category_dict['items'].append(item_dict)
    structured_menu_items['menuItems'].append(category_dict)

print(json.dumps(structured_menu_items, indent=2))


## 7: Data Preparation and Embedding Generation

### Description
This cell defines functions to sanitize document keys, generate embeddings using Azure OpenAI, and prepare the Sonic Drive-In menu data for ingestion into Azure AI Search. It includes retry logic for embedding generation and assigns embeddings to each document.


In [ ]:
def sanitize_key(key):
    """Sanitize the document key to contain only valid characters."""
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', key)

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=5, max=60))
def generate_embeddings(texts):
    """Generate embeddings using Azure OpenAI with retry logic for a batch of texts."""
    response = aoai_client.embeddings.create(input=texts, model=aoai_embedding_deployment)
    return [res.embedding for res in response.data]

def prepare_data_for_azure_search(menu_items_data):
    """Transform parsed Sonic Drive-In menu data for ingestion into Azure AI Search."""
    azure_search_documents = []
    menu_items = menu_items_data['menuItems']
    texts_for_embedding = []
    document_keys = []

    for category in menu_items:
        for item in category['items']:
            combined_text = f"{category['category']} {item['name']} {item['description']} {item.get('longDescription', '')}"
            document_key = sanitize_key(f"{category['category']}_{item['name'].replace(' ', '_')}".lower())

            texts_for_embedding.append(combined_text)
            document_keys.append(document_key)

            azure_search_documents.append({
                'id': document_key,
                'category': category['category'],
                'name': item['name'],
                'description': item['description'],
                'longDescription': item.get('longDescription', ''),
                'origin': item.get('origin', ''),
                'caffeineContent': item.get('caffeineContent', ''),
                'brewingMethod': item.get('brewingMethod', ''),
                'popularity': item.get('popularity', ''),
                'sizes': json.dumps(item['sizes']),
            })

    # Generate embeddings in batch
    embeddings = generate_embeddings(texts_for_embedding)

    # Assign embeddings to documents
    for i, embedding in enumerate(embeddings):
        azure_search_documents[i]['embedding'] = embedding

    return azure_search_documents

documents_for_index = prepare_data_for_azure_search(structured_menu_items)

for doc in documents_for_index:
    print(f"ID: {doc['id']}")
    print(f"Category: {doc['category']}")
    print(f"Name: {doc['name']}")
    print(f"Description: {doc['description']}")
    print(f"Long Description: {doc['longDescription']}")
    print(f"Origin: {doc['origin']}")
    print(f"Caffeine Content: {doc['caffeineContent']}")
    print(f"Brewing Method: {doc['brewingMethod']}")
    print(f"Popularity: {doc['popularity']}")
    print(f"Sizes: {doc['sizes']}")
    print(f"Embedding: {doc['embedding'][:10]}...")
    print()


## 8: Upload to Azure AI Search

### Description
This cell uploads the prepared Sonic Drive-In menu data to Azure AI Search.
Ensure the Azure AI Search index is properly configured before running this step.


In [ ]:
def upload_documents_to_search(documents):
    batch_size = 15
    total_batches = (len(documents) + batch_size - 1) // batch_size
    successful_uploads = 0

    for i in range(0, len(documents), batch_size):
        batch = documents[i:i + batch_size]
        try:
            response = search_client.upload_documents(documents=batch)
            successful_uploads += len(batch)
            print(f'Uploaded batch {i // batch_size + 1}/{total_batches} successfully. Batch size: {len(batch)}')
        except HttpResponseError as e:
            print(f'Error uploading batch {i // batch_size + 1}/{total_batches}: {e}')
            continue

    print(f'Sonic menu items uploaded successfully. Total: {successful_uploads}/{len(documents)}')

upload_documents_to_search(documents_for_index)


## 9: Testing Search Capabilities

### Summary
Test the Sonic Drive-In menu search index with various query types:
- Simple text-based search for "burger"
- Semantic search for "frozen drinks with fruit flavors"
- Vector search for "crispy chicken sandwich with pickles"
- Hybrid search combining text and vector for "cold refreshing slush"
- Filter-based search for items in the "Shakes" category


In [ ]:
# Test Azure AI Search with various query types
%pip install --upgrade azure-search-documents
from azure.search.documents.models import VectorizedQuery


def perform_simple_search(query_text, filter_condition=None, top=5):
    """Perform a simple text-based search against the Sonic menu index"""
    results = search_client.search(
        search_text=query_text,
        filter=filter_condition,
        select=['id', 'name', 'description', 'category', 'origin'],
        top=top,
        include_total_count=True
    )

    print(f"\n=== Simple Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Origin: {result.get('origin', 'Not specified')}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


def perform_semantic_search(query_text, top=5):
    """Perform a semantic search using the configured semantic settings"""
    try:
        results = search_client.search(
            search_text=query_text,
            select=['id', 'name', 'description', 'category', 'origin'],
            top=top,
            query_type='semantic',
            semantic_configuration_name='menuSemanticConfig',
            include_total_count=True
        )
    except HttpResponseError as e:
        print('Semantic search is not enabled for this search service. Enable it in the Azure portal or use simple search instead.')
        print(f'Details: {e}')
        return None

    print(f"\n=== Semantic Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Origin: {result.get('origin', 'Not specified')}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


def perform_vector_search(query_text, top=5):
    """Generate embedding for the query and perform vector search"""
    client = AzureOpenAI(
        api_key=aoai_eastus_api_key,
        api_version='2023-05-15',
        azure_endpoint=aoai_eastus_endpoint
    )

    response = client.embeddings.create(input=query_text, model=aoai_embedding_deployment)
    query_vector = response.data[0].embedding

    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=top, fields='embedding')
    results = search_client.search(
        search_text=None,
        select=['id', 'name', 'description', 'category', 'origin'],
        top=top,
        vector_queries=[vector_query],
        include_total_count=True
    )

    print(f"\n=== Vector Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Origin: {result.get('origin', 'Not specified')}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


def perform_hybrid_search(query_text, top=5):
    """Perform a hybrid search combining vector search with text search"""
    client = AzureOpenAI(
        api_key=aoai_eastus_api_key,
        api_version='2023-05-15',
        azure_endpoint=aoai_eastus_endpoint
    )

    response = client.embeddings.create(input=query_text, model=aoai_embedding_deployment)
    query_vector = response.data[0].embedding

    vector_query = VectorizedQuery(vector=query_vector, k_nearest_neighbors=top, fields='embedding')
    results = search_client.search(
        search_text=query_text,
        select=['id', 'name', 'description', 'category', 'origin'],
        top=top,
        vector_queries=[vector_query],
        include_total_count=True
    )

    print(f"\n=== Hybrid Search Results for '{query_text}' ===")
    print(f'Total matches: {results.get_count()}')

    for result in results:
        print(f"\nName: {result['name']}")
        print(f"Category: {result['category']}")
        print(f"Origin: {result.get('origin', 'Not specified')}")
        print(f"Description: {result['description']}")
        print('-' * 50)

    return results


# Test different search types against the Sonic Drive-In menu
print('\n========= TESTING SONIC DRIVE-IN SEARCH CAPABILITIES =========')

# Test 1: Simple text search
simple_results = perform_simple_search('burger')

# Test 2: Semantic search for natural language queries
semantic_results = perform_semantic_search('frozen drinks with fruit flavors')

# Test 3: Vector search for semantic similarity
vector_results = perform_vector_search('crispy chicken sandwich with pickles')

# Test 4: Hybrid search combining text and vector capabilities
hybrid_results = perform_hybrid_search('cold refreshing slush')

# Test 5: Search with category filter
filtered_results = perform_simple_search(
    'shake',
    filter_condition="category eq 'Shakes'"
)


## 10: Final Summary

### Summary
- Installed required packages.
- Configured Azure OpenAI and Azure AI Search services.
- Defined and created the `sonic-menu-items` index schema with semantic configuration.
- Loaded and processed Sonic Drive-In menu data from `sonicmenuitems.json`.
- Prepared data for ingestion, including generating embeddings using Azure OpenAI.
- Uploaded the structured Sonic Drive-In menu data into Azure AI Search.
- Tested search capabilities with Sonic-relevant queries (burgers, slushes, shakes, etc.).

The Sonic Drive-In menu ingestion pipeline is complete!
